# Human Expert Validation

This notebook processes expert validation surveys for AgentSLR data extractions.

## Methodology

We recruited epidemiologists to grade AgentSLR-generated data extractions through form submissions. Each expert committed up to 10 hours over 1 to 2 weeks. Assignments considered pathogen expertise and familiarity with SLR workflows.

For each data type (parameters, models, outbreaks) and pathogen (Lassa, Ebola, SARS, Zika), we randomly sampled screened articles until subsamples exceeded each expert's time commitment. Different articles contain varying numbers of extractions requiring different grading times, so parity across pathogens or data types is not guaranteed.

Experts received a private GitHub repository containing Markdown extractions from our OCR model alongside readable renderings of the structured data. Submissions were collected via Google Forms with the following structure:

1. Article identifier, pathogen identifier, and pathogen name
2. Assessment of Markdown document quality affecting data extraction
3. Determination of whether extractable data exists (before viewing AgentSLR output)
4. Overall relevance rating for the extraction
5. Yes/no validation questions for each extracted field
6. Likert scale competence rating (1 to 7), where:
   - 1: System gets nothing right; cannot speed up my process
   - 4: System identifies some things but struggles with edge cases; usable with moderate supervision
   - 7: System is perfectly capable of doing all extraction for me
7. Self-reported time estimate for survey completion

## Setup

In [1]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/human_validation")

## Models

In [2]:
models_survey = pd.read_csv(DATA_DIR / "models" / "models_survey.csv")
models_survey = models_survey[models_survey["Username"] != "padarhas@gmail.com"]
print(f"Models survey responses: {models_survey.shape[0]}")

Models survey responses: 50


In [3]:
models_survey

,Timestamp,Username,Article Covidence ID,Pathogen,"Markdown document\n\nPlease open the Markdown document for the article (articles/{covidence_id}.md). Our system transforms article PDFs into Markdown to make them machine readable, but this transformation process might be error-prone.\n\nAre there any issues with the Markdown that would affect data extraction?","If yes, what issues?","Model ID\nThis is present below the ""Select Model"" dropdown as ""Model ID:"".","Was the model extracted (with ID:XXX), an extractable model ?",If No:\nWhy should the model have not extracted this model?,\nWas 'Model Type' correctly extracted?,...,"Are there any more models left to extract (see ""Select Model"" dropdown)","Overall AI Competence\n\nOverall, how competent was the AI system at model extraction for this article?",How long do you think it took you to complete this survey?,Notes (optional)\nAre there any other remarks that you would like to leave?,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32
1,2026/01/18 5:55:41 PM GMT,t.naidoo21@imperial.ac.uk,5423,Lassa,"No, no significant issues",NaN,1,"No, the model has made an incorrect extraction.",Phylogenetic analysis - there is no transmissi...,NaN,...,No,1,5 minutes,Rating of 1 is becuase there was no model to e...,NaN,NaN,NaN,NaN,NaN,NaN
2,2026/01/19 3:00:43 PM GMT,t.naidoo21@imperial.ac.uk,4316,Lassa,"No, no significant issues",NaN,1,"Yes, it should have been extracted.",NaN,Yes,...,No,4,10 minutes,"- - By having an SEIR-SEI model, I think spill...",Field should be NULL (not applicable),Yes,Field should be NULL (not applicable),NaN,NaN,NaN
3,2026/01/19 3:15:00 PM GMT,t.naidoo21@imperial.ac.uk,4144,Lassa,"No, no significant issues",NaN,1,"Yes, it should have been extracted.",NaN,Yes,...,No,6,10 minutes,The LLM is extracting information from the wro...,Field should be NULL (not applicable),Yes,Field should be NULL (not applicable),NaN,NaN,NaN
4,2026/01/19 5:08:34 PM GMT,t.naidoo21@imperial.ac.uk,3735,Lassa,"No, no significant issues",NaN,1,"Yes, it should have been extracted.",NaN,Yes,...,No,5,10 minutes,Compartmental Type: AI-evidence is incorrect (...,Field should be NULL (not applicable),Yes,Yes,NaN,NaN,NaN
5,2026/01/19 5:28:12 PM GMT,t.naidoo21@imperial.ac.uk,558,Lassa,"No, no significant issues",NaN,1,"Yes, it should have been extracted.",NaN,Yes,...,Yes,6,20,"For assumptions, based on the AI-evidence I un...",Field should be NULL (not applicable),Yes,Yes,NaN,NaN,NaN
6,2026/01/19 5:59:08 PM GMT,t.naidoo21@imperial.ac.uk,23719,Ebola,"No, no significant issues",NaN,1,"Yes, it should have been extracted.",NaN,Yes,...,No,7,10 minutes,"A really good extraction, a longer AI evidence...",Field should be NULL (not applicable),Field should be NULL (not applicable),Yes,NaN,NaN,NaN
7,2026/01/19 6:00:22 PM GMT,t.naidoo21@imperial.ac.uk,558,Lassa,"No, no significant issues",NaN,3,"No, the model has made an incorrect extraction.",NaN,NaN,...,Yes,1,30 seconds,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026/01/19 6:01:50 PM GMT,t.naidoo21@imperial.ac.uk,558,Lassa,"No, no significant issues",NaN,4,"No, the model has made an incorrect extraction.",A repetition of model 1 (the branching process...,NaN,...,Yes,1,30 seconds,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026/01/19 6:02:29 PM GMT,t.naidoo21@imperial.ac.uk,558,Lassa,"No, no significant issues",NaN,5,"No, the model has made an incorrect extraction.",A repetition of model 1 (the branching process...,NaN,...,Yes,1,30 seconds,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2026/01/19 6:03:07 PM GMT,t.naidoo21@imperial.ac.uk,558,Lassa,"No, no significant issues",NaN,6,"No, the model has made an incorrect extraction.",A repetition of model 1 (the branching process...,NaN,...,No,1,30 seconds,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
models_flagging_precision = (
    models_survey["Was the model extracted (with ID:XXX), an extractable model ?"] 
    == "Yes, it should have been extracted."
).mean()
print(f"Models flagging precision: {models_flagging_precision:.3f}")

Models flagging precision: 0.400


In [5]:
models_question_to_field = {
    "\nWas 'Model Type' correctly extracted?": "model_type",
    "Was 'Compartmental Type' correctly extracted?": "compartmental_type",
    "Was 'Stochastic/Deterministic' correctly extracted?": "stoch_deter",
    "Was 'Transmission Routes' correctly extracted?": "transmission_routes",
    "Was 'Assumptions' correctly extracted?": "assumptions",
    "Was 'Theoretical Model' correctly extracted?": "theoretical_model",
    "Was 'Intervention Types' correctly extracted?": "intervention_types",
}

answer_mapping = {"Yes": True, "No": False, pd.NA: pd.NA}

for question, field in models_question_to_field.items():
    models_survey[field] = models_survey[question].map(answer_mapping)

models_field_accuracies = {}
for field in models_question_to_field.values():
    models_field_accuracies[field] = {
        "accuracy": models_survey[field].dropna().mean(),
        "n": models_survey[field].dropna().shape[0],
    }

models_field_df = pd.DataFrame(models_field_accuracies).T
models_weighted_acc = (
    (models_field_df["accuracy"] * models_field_df["n"]).sum() 
    / models_field_df["n"].sum()
)
print(f"Models weighted accuracy: {models_weighted_acc:.3f}")
models_field_df

Models weighted accuracy: 0.829


,accuracy,n
model_type,0.894737,19.0
compartmental_type,0.888889,18.0
stoch_deter,0.700000,20.0
transmission_routes,NaN,0.0
assumptions,NaN,0.0
theoretical_model,0.842105,19.0
intervention_types,NaN,0.0


In [6]:
models_competence_col = "Overall AI Competence\n\nOverall, how competent was the AI system at model extraction for this article?"
models_competence_ratings = models_survey[models_competence_col].values.tolist()
print(f"Models mean competence: {pd.Series(models_competence_ratings).mean():.3f}")

Models mean competence: 2.800


In [7]:
models_output = {
    "flagging": {"precision": models_flagging_precision},
    "field_accuracies": {
        "average": models_weighted_acc,
        "fields": models_field_df["accuracy"].dropna().to_dict(),
    },
    "competence_ratings": models_competence_ratings,
}

with open(DATA_DIR / "models" / "results.json", "w") as f:
    json.dump(models_output, f, indent=4)

## Outbreaks

In [8]:
outbreaks_survey = pd.read_csv(DATA_DIR / "outbreaks" / "outbreaks_survey.csv")
print(f"Outbreaks survey responses: {outbreaks_survey.shape[0]}")

Outbreaks survey responses: 31


In [9]:
outbreaks_flagging_precision = (
    outbreaks_survey["Was the outbreak extracted (with ID:XXX), an extractable outbreak ?"] 
    == "Yes, it should have been extracted."
).mean()
print(f"Outbreaks flagging precision: {outbreaks_flagging_precision:.3f}")

Outbreaks flagging precision: 0.613


In [10]:
outbreaks_question_to_field = {
    "Start Year\nWas the correct Outbreak 'Start Year' extracted ?": "outbreak_start_year",
    "Start Month\nWas the correct Outbreak 'Start Month' extracted ?": "outbreak_start_month",
    "Start Day\nWas the correct Outbreak 'Start Day' extracted ?": "outbreak_start_day",
    "End Year\nWas the correct Outbreak 'End Year' extracted ?": "outbreak_end_year",
    "End Month\nWas the correct Outbreak 'End Month' extracted ?": "outbreak_end_month",
    "End Day\nWas the correct Outbreak 'End Day' extracted ?": "outbreak_end_day",
    "Duration (Months)\nWas the correct Outbreak 'Duration (Months)' extracted ?": "outbreak_duration_months",
    "Country\nWas the correct 'Country' extracted ?": "outbreak_country",
    "Location\nWas the correct 'Location' extracted ?": "outbreak_location",
    "Confirmed Cases\nWere 'Confirmed Cases' correctly extracted?": "cases_confirmed",
    "Suspected Cases\nWere 'Suspected Cases' correctly extracted?": "cases_suspected",
    "Asymptomatic Cases\nWere 'Asymptomatic Cases' correctly extracted?": "cases_asymptomatic",
    "Severe Cases\nWere 'Severe Cases' correctly extracted?": "cases_severe",
    "Deaths\nWere 'Deaths' correctly extracted?": "deaths",
    "Mode of Detection\nWas 'Mode of Detection' correctly extracted?": "mode_of_detection",
    "Pre-outbreak Status\nWas 'Pre-outbreak Status' correctly extracted?": "pre_outbreak",
    "Asymptomatic Transmission Described\nWas 'Asymptomatic Transmission Described' correctly extracted?": "asymptomatic_transmission_described",
}

outbreaks_field_to_group = {
    "outbreak_start_year": "temporal",
    "outbreak_start_month": "temporal",
    "outbreak_start_day": "temporal",
    "outbreak_end_year": "temporal",
    "outbreak_end_month": "temporal",
    "outbreak_end_day": "temporal",
    "outbreak_duration_months": "temporal",
    "outbreak_country": "geographical",
    "outbreak_location": "geographical",
    "cases_confirmed": "case_burden",
    "cases_suspected": "case_burden",
    "cases_asymptomatic": "case_burden",
    "cases_severe": "case_burden",
    "deaths": "case_burden",
    "mode_of_detection": "epidemiological",
    "pre_outbreak": "epidemiological",
    "asymptomatic_transmission_described": "epidemiological",
}

for question, field in outbreaks_question_to_field.items():
    outbreaks_survey[field] = outbreaks_survey[question].map(answer_mapping)

outbreaks_field_accuracies = {}
for field in outbreaks_question_to_field.values():
    outbreaks_field_accuracies[field] = {
        "accuracy": outbreaks_survey[field].dropna().mean(),
        "n": outbreaks_survey[field].dropna().shape[0],
    }

outbreaks_field_df = pd.DataFrame(outbreaks_field_accuracies).T
outbreaks_field_df["group"] = outbreaks_field_df.index.map(outbreaks_field_to_group)

outbreaks_group_accuracies = outbreaks_field_df.groupby("group")[["accuracy"]].mean().to_dict()["accuracy"]
outbreaks_normalized_accuracy = outbreaks_field_df.groupby("group")["accuracy"].mean().mean()

print(f"Outbreaks normalised accuracy: {outbreaks_normalized_accuracy:.3f}")
outbreaks_field_df

Outbreaks normalised accuracy: 0.796


,accuracy,n,group
outbreak_start_year,0.842105,19.0,temporal
outbreak_start_month,0.700000,10.0,temporal
outbreak_start_day,0.625000,8.0,temporal
outbreak_end_year,0.500000,18.0,temporal
outbreak_end_month,0.600000,10.0,temporal
outbreak_end_day,0.555556,9.0,temporal
outbreak_duration_months,0.500000,2.0,temporal
outbreak_country,0.947368,19.0,geographical
outbreak_location,0.800000,15.0,geographical
cases_confirmed,0.882353,17.0,case_burden


In [11]:
outbreaks_competence_col = "Overall AI Competence\n\nOverall, how competent was the AI system at outbreak extraction for this article?"
outbreaks_competence_ratings = outbreaks_survey[outbreaks_competence_col].values.tolist()
print(f"Outbreaks mean competence: {pd.Series(outbreaks_competence_ratings).mean():.3f}")

Outbreaks mean competence: 3.903


In [12]:
outbreaks_field_dict = outbreaks_field_df[["accuracy", "group"]].to_dict(orient="index")

outbreaks_output = {
    "flagging": {"precision": outbreaks_flagging_precision},
    "field_accuracies": {
        "average": outbreaks_normalized_accuracy,
        "groups": outbreaks_group_accuracies,
        "fields": outbreaks_field_dict,
    },
    "competence_ratings": outbreaks_competence_ratings,
}

with open(DATA_DIR / "outbreaks" / "results.json", "w") as f:
    json.dump(outbreaks_output, f, indent=4)

## Parameters

In [13]:
parameters_survey = pd.read_csv(DATA_DIR / "parameters" / "parameters_survey.csv")
articles_to_exclude = ["3624", "3703", "4326"]
parameters_survey = parameters_survey[
    ~parameters_survey["Article Covidence ID"].isin(articles_to_exclude)
]
print(f"Parameters survey responses: {parameters_survey.shape[0]}")

Parameters survey responses: 62


In [14]:
# Parse expert parameter flags from survey
parameters_expert = parameters_survey["What parameters does this paper have?"].str.split(r";(?=\S)").explode()
parameters_dummies = pd.get_dummies(parameters_expert).groupby(level=0).max().astype(bool)
parameters_survey_flagging = parameters_survey.join(parameters_dummies)

# Combine CFR and proportion of symptomatic cases into "severity"
if "Case fatality rate (CFR)" in parameters_survey_flagging.columns or "Proportion of symptomatic cases" in parameters_survey_flagging.columns:
    cfr_col = parameters_survey_flagging.get("Case fatality rate (CFR)", pd.Series(False, index=parameters_survey_flagging.index))
    symptomatic_col = parameters_survey_flagging.get("Proportion of symptomatic cases", pd.Series(False, index=parameters_survey_flagging.index))
    parameters_survey_flagging["severity"] = cfr_col.fillna(False) | symptomatic_col.fillna(False)
    parameters_survey_flagging.drop(columns=["Case fatality rate (CFR)", "Proportion of symptomatic cases"], inplace=True, errors="ignore")

# Map survey parameter names to system parameter class names
parameter_name_mapping = {
    "Attack rate (including secondary attack rate)": "attack_rate",
    "Growth rate (r)": "growth_rate",
    "Human delays (incubation period; latent period; etc.)": "human_delay",
    "Overdispersion": "overdispersion",
    "Reproduction numbers (basic R0; effective Re)": "reproduction_number",
    "Risk factors": "risk_factors",
    "Seroprevance": "seroprevalence",
}
parameters_survey_flagging.rename(columns=parameter_name_mapping, inplace=True)

In [15]:
# Load system flagging results from screening.jsonl files
flagging_results_paths = {
    "lassa": DATA_DIR / "parameters/parameter_run_logs_original/2026-01-18_14-34-39_lassa_from_script/lassa/results/screening.jsonl",
    "sars": DATA_DIR / "parameters/parameter_run_logs_original/2026-01-18_17-49-42_sars_from_script/sars/results/screening.jsonl",
    "ebola": DATA_DIR / "parameters/parameter_run_logs_original/2026-01-18_15-14-47_ebola_from_script/ebola/results/screening.jsonl",
    "zika": DATA_DIR / "parameters/parameter_run_logs_original/2026-01-18_11-36-42_zika_from_script/zika/results/screening.jsonl",
}

our_flagging_results = pd.DataFrame()

for pathogen, path in flagging_results_paths.items():
    if not path.exists():
        print(f"Warning: {path} does not exist, skipping {pathogen}")
        continue
    results = pd.read_json(path, lines=True)
    results = results[results["covidence_id"].astype(str).isin(parameters_survey["Article Covidence ID"].astype(str))]
    if results.empty:
        continue
    results = results[results["excerpts"].apply(lambda x: x != [])]
    results = results[["covidence_id", "parameter_class"]]
    our_flagging_results = pd.concat([our_flagging_results, results])

if not our_flagging_results.empty:
    our_flagging_results.set_index("covidence_id", inplace=True)
    our_flagging_results.index = our_flagging_results.index.astype(str)
    our_flagging_dummies = pd.get_dummies(our_flagging_results["parameter_class"]).groupby(level=0).max().astype(bool)
    our_flagging_results = our_flagging_dummies

print(f"System flagging results loaded for {len(our_flagging_results)} articles")

System flagging results loaded for 12 articles


In [16]:
# Compute flagging precision per parameter class
flagging_metrics = {}

for parameter_class in our_flagging_results.columns:
    if parameter_class not in parameters_survey_flagging.columns:
        continue
    
    expert_validated_flags = parameters_survey_flagging.set_index("Article Covidence ID")[[parameter_class]]
    expert_validated_flags.index = expert_validated_flags.index.astype(str)
    our_flags = our_flagging_results[[parameter_class]]

    comparison = expert_validated_flags.merge(
        our_flags, left_index=True, right_index=True, how="inner", suffixes=("_expert", "_ours")
    )
    comparison = comparison[~comparison.index.duplicated(keep="first")]

    tp = ((comparison[f"{parameter_class}_expert"] == True) & (comparison[f"{parameter_class}_ours"] == True)).sum()
    fp = ((comparison[f"{parameter_class}_expert"] == False) & (comparison[f"{parameter_class}_ours"] == True)).sum()
    fn = ((comparison[f"{parameter_class}_expert"] == True) & (comparison[f"{parameter_class}_ours"] == False)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    flagging_metrics[parameter_class] = {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }

flagging_metrics_df = pd.DataFrame(flagging_metrics).T
parameters_flagging_precision = flagging_metrics_df["precision"].mean()
print(f"Parameters flagging precision: {parameters_flagging_precision:.3f}")
flagging_metrics_df

Parameters flagging precision: 0.658


,precision,recall,f1_score
attack_rate,0.250000,1.000000,0.400000
growth_rate,1.000000,1.000000,1.000000
human_delay,0.625000,0.833333,0.714286
reproduction_number,1.000000,1.000000,1.000000
seroprevalence,0.500000,1.000000,0.666667
severity,0.571429,0.800000,0.666667


In [17]:
parameters_question_to_field = {
    "Does the extraction have the correct value?": "value",
    "Does the extraction have the correct unit?": "unit",
    "Does the extraction have the correct type?": "type",
    "Does the extraction use the rule of three appropriately?": "aggregation",
    "Does the extraction have the correct upper and lower bounds?": "bounds",
    "Does the extraction have the correct value type?": "value_type",
    "Does the extraction have the correct statistical approach?": "statistical_approach",
    "Does the extraction have the correct single type uncertainty?": "single_type_uncertainty",
    "Does the extraction have the correct paired uncertainty?": "paired_uncertainty",
    "Does the extraction have the correct distribution type?": "distribution_type",
    "Does the extraction have the correct population sample type?": "population_sample_type",
    "Does the extraction have the correct population group?": "population_group",
    "Does the extraction have the correct population sample size?": "population_sample_size",
    "Does the extraction have the correct population sex?": "population_sex",
    "Does the extraction have the correct population age range?": "population_age_range",
    "Does the extraction have the correct population countries?": "population_countries",
    "Does the extraction have the correct population locations?": "population_locations",
    "Does the extraction have the correct method moment value?": "method_moment_value",
    "Does the extraction have the correct genome site?": "genome_site",
    "Does the extraction have the correct transmission?": "transmission",
    "Does the extraction have the correct method?": "method",
}

parameters_field_to_group = {
    "value": "value",
    "unit": "value",
    "type": "value",
    "aggregation": "aggregation",
    "bounds": "value",
    "value_type": "value",
    "statistical_approach": "value",
    "single_type_uncertainty": "uncertainty",
    "paired_uncertainty": "uncertainty",
    "distribution_type": "uncertainty",
    "population_sample_type": "population",
    "population_group": "population",
    "population_sample_size": "population",
    "population_sex": "population",
    "population_age_range": "population",
    "population_countries": "population",
    "population_locations": "population",
    "method_moment_value": "population",
    "genome_site": "metadata",
    "transmission": "metadata",
    "method": "metadata",
}

for question, field in parameters_question_to_field.items():
    parameters_survey[field] = parameters_survey[question].map(answer_mapping)

parameters_field_accuracies = {}
for field in parameters_question_to_field.values():
    parameters_field_accuracies[field] = {
        "accuracy": parameters_survey[field].dropna().mean(),
        "n": parameters_survey[field].dropna().shape[0],
    }

parameters_field_df = pd.DataFrame(parameters_field_accuracies).T
parameters_field_df["group"] = parameters_field_df.index.map(parameters_field_to_group)

parameters_field_df_filtered = parameters_field_df[parameters_field_df["group"] != "metadata"]
parameters_group_accuracies = parameters_field_df_filtered.groupby("group")[["accuracy"]].mean().to_dict()["accuracy"]
parameters_normalized_accuracy = parameters_field_df_filtered.groupby("group")["accuracy"].mean().mean()

print(f"Parameters normalised accuracy: {parameters_normalized_accuracy:.3f}")
parameters_field_df

Parameters normalised accuracy: 0.768


,accuracy,n,group
value,0.807692,52.0,value
unit,0.960784,51.0,value
type,0.880952,42.0,value
aggregation,0.833333,42.0,aggregation
bounds,0.789474,38.0,value
value_type,0.904762,42.0,value
statistical_approach,0.971429,35.0,value
single_type_uncertainty,0.875000,16.0,uncertainty
paired_uncertainty,0.842105,38.0,uncertainty
distribution_type,0.571429,14.0,uncertainty


In [18]:
parameters_competence_col = "Overall, how competent was the AI system at parameter extraction for this article?"
parameters_competence_ratings = parameters_survey[parameters_competence_col].values.tolist()
print(f"Parameters mean competence: {pd.Series(parameters_competence_ratings).mean():.3f}")

Parameters mean competence: 4.226


In [19]:
parameters_field_dict = parameters_field_df[["accuracy", "group"]].to_dict(orient="index")

parameters_output = {
    "flagging": {"precision": parameters_flagging_precision},
    "field_accuracies": {
        "average": parameters_normalized_accuracy,
        "groups": parameters_group_accuracies,
        "fields": parameters_field_dict,
    },
    "competence_ratings": parameters_competence_ratings,
}

with open(DATA_DIR / "parameters" / "results.json", "w") as f:
    json.dump(parameters_output, f, indent=4)

## Combined Results

In [20]:
combined_output = {
    "parameters": parameters_output,
    "models": models_output,
    "outbreaks": outbreaks_output,
}

with open(DATA_DIR / "results.json", "w") as f:
    json.dump(combined_output, f, indent=4)

print("Results saved to data/human_validation/results.json")

Results saved to data/human_validation/results.json
